In [ ]:
import csv
import os
import glob # Module to find files matching a pattern

def remove_rows_with_null_question(input_filepath, output_filepath, question_column_name='Question'):
    """
    Reads a CSV file, removes rows where the specified question column is null or empty,
    and writes the cleaned data to a new CSV file.

    Args:
        input_filepath (str): The path to the input CSV file.
        output_filepath (str): The path where the cleaned CSV file will be saved.
        question_column_name (str): The exact name of the column containing the questions.
                                    Defaults to 'Question'.
    """
    print(f"--- Processing file: {os.path.basename(input_filepath)} ---")
    rows_processed = 0
    rows_removed = 0
    try:
        with open(input_filepath, 'r', newline='', encoding='utf-8') as infile, \
             open(output_filepath, 'w', newline='', encoding='utf-8') as outfile:

            reader = csv.reader(infile)
            writer = csv.writer(outfile)

            # Read the header row
            try:
                header = next(reader)
            except StopIteration:
                print(f"  Info: Input file '{os.path.basename(input_filepath)}' is empty. Skipping.")
                return # Skip if the file is empty

            writer.writerow(header) # Write header to output file

            # Find the index of the question column
            try:
                question_col_index = header.index(question_column_name)
            except ValueError:
                print(f"  Error: Column '{question_column_name}' not found in the header of '{os.path.basename(input_filepath)}'. Skipping this file.")
                print(f"  Header found: {header}")
                # Optionally remove the potentially empty output file created
                try:
                    outfile.close() # Close the file first
                    os.remove(output_filepath)
                except OSError as e:
                    print(f"  Warning: Could not remove empty output file '{output_filepath}': {e}")
                return # Skip this file if the column doesn't exist

            # Process the rest of the rows
            for i, row in enumerate(reader, start=1): # Start counting rows from 1 after header
                rows_processed += 1
                # Check if the row has enough columns before accessing the index
                if len(row) > question_col_index:
                    question_value = row[question_col_index]
                    # Check if the question value is considered null or empty
                    # You might want to expand this check based on your specific null values (e.g., 'NA', 'None', 'null')
                    if question_value is not None and question_value.strip() != '':
                        writer.writerow(row)
                    else:
                        rows_removed += 1
                else:
                    # Handle rows that are shorter than expected (optional: log or skip)
                    print(f"  Warning: Row {i} in '{os.path.basename(input_filepath)}' has fewer columns than expected. Skipping row.")
                    rows_removed += 1 # Count it as removed if it can't be processed

            print(f"  Processing complete for '{os.path.basename(input_filepath)}'.")
            print(f"  Total data rows processed: {rows_processed}")
            print(f"  Rows removed due to empty/null '{question_column_name}': {rows_removed}")
            print(f"  Cleaned data saved to: '{output_filepath}'")

    except FileNotFoundError:
        print(f"Error: Input file not found at '{input_filepath}' (This shouldn't happen if glob found it).")
    except Exception as e:
        print(f"An unexpected error occurred while processing '{os.path.basename(input_filepath)}': {e}")
        # Optionally remove partially written output file on error
        try:
            outfile.close()
            os.remove(output_filepath)
            print(f"  Removed potentially incomplete output file: '{output_filepath}'")
        except Exception as e_rem:
             print(f"  Warning: Could not remove output file '{output_filepath}' after error: {e_rem}")


# --- Main execution block for Colab ---
if __name__ == "__main__":
    # --- Configuration ---
    # Set the path to the folder containing your input CSV files in Colab
    # Option 1: If you upload a 'data' folder directly to the Colab session storage
    input_folder = '/content/data/'
    # Option 2: If you mount Google Drive and your 'data' folder is in the root of 'My Drive'
    # input_folder = '/content/drive/MyDrive/data/'
    # Option 3: If your 'data' folder is elsewhere in Drive, adjust the path accordingly
    # input_folder = '/content/drive/MyDrive/path/to/your/data/'

    # Set the path for the folder where cleaned CSV files will be saved
    # It's good practice to save to a different folder
    output_folder = '/content/cleaned_data/'
    # Or save to Drive:
    # output_folder = '/content/drive/MyDrive/cleaned_data/'

    # Set the name of the column to check for null/empty values
    question_col_name = 'Question'
    # --- End Configuration ---


    print(f"Starting CSV cleaning process...")
    print(f"Input folder: {input_folder}")
    print(f"Output folder: {output_folder}")
    print(f"Column to check: '{question_col_name}'")

    # Create the output directory if it doesn't exist
    os.makedirs(output_folder, exist_ok=True) # exist_ok=True prevents error if folder already exists
    print(f"Ensured output directory exists: {output_folder}")

    # Find all CSV files in the input folder
    # The pattern '*.csv' matches any file ending with .csv
    csv_files = glob.glob(os.path.join(input_folder, '*.csv'))

    if not csv_files:
        print(f"\nWarning: No CSV files found in the specified input folder: '{input_folder}'")
        print("Please ensure the path is correct and the folder contains .csv files.")
    else:
        print(f"\nFound {len(csv_files)} CSV files to process:")
        # Process each CSV file found
        for input_file_path in csv_files:
            # Construct the output file path
            # Takes the original filename, adds 'cleaned_' prefix, and puts it in the output folder
            base_filename = os.path.basename(input_file_path)
            output_file_path = os.path.join(output_folder, f"cleaned_{base_filename}")

            # Run the cleaning function for the current file
            remove_rows_with_null_question(input_file_path, output_file_path, question_col_name)
            print("-" * 20) # Separator between files

    print("\nScript finished.")

Starting CSV cleaning process...
Input folder: /content/data/
Output folder: /content/cleaned_data/
Column to check: 'Question'
Ensured output directory exists: /content/cleaned_data/

Found 1 CSV files to process:
--- Processing file: output (4).csv ---
  Processing complete for 'output (4).csv'.
  Total data rows processed: 3787
  Rows removed due to empty/null 'Question': 968
  Cleaned data saved to: '/content/cleaned_data/cleaned_output (4).csv'
--------------------

Script finished.
